## RPCA

In this notebook, RPCA is used for facial reconstruction of the occluded images. 

The difference between image reconstruction with a CNN Unet and RPCA is that RPCA does not use "training" with ground truth clear images in order to learn representation and apply it to unseen faces. 

RPCA functions as an unsupervised model. It does not use ground truth original images. The matrix fed into it consists of only training images with sparse occlusion, and the structure of faces is learned from redundancy from the parts of the images that aren't occluded. This is why for RPCA, it is important that occlusion is random across images and not occluding the same part of every image so that RPCA does not learn that the occlusion is part of the low rank face but instead recognizes it as sparse noise.
Then the learned subspace from the training images is applied to the testing occluded images to reconstruct them.
 

In [35]:
#import packages
import os
import imageio
import numpy as np
from sklearn.model_selection import train_test_split
import time
from numpy.linalg import norm, svd

In [8]:
#import images (coppied from Unet improvements notebook)

# Create the array of image vectors. 

dataset_path = "CroppedYaleCopy/CroppedYale Project Data/CroppedYale"

images = [] # To store the images 
labels = [] # To store the labels. 

for subject in sorted(os.listdir(dataset_path)):
    
    subject_path = os.path.join(dataset_path, subject)
    
    if not os.path.isdir(subject_path):
        continue
        
    for file in os.listdir(subject_path):
        
        # Skip ambient images if any have been detected. 
        if "Ambient" in file:
            continue
            
        if file.endswith(".pgm"):
            
            path = os.path.join(subject_path, file)
            img = imageio.imread(path)
            
            images.append(img) # Append images. 
            labels.append(subject) # Append label to be the number of the person whose image it is. 

X = np.array(images) # Comvert to array.
y = np.array(labels) # Convert labels to array

print("Dataset shape:", X.shape)

/tmp/ipykernel_967/710943570.py:26: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(path)


Dataset shape: (2414, 192, 168)


In [9]:
#normalize the image vectors in the same way as with U-net 

X = X.astype("float32") 
X = (X - X.min()) / (X.max() - X.min()) 
print(X.shape)

(2414, 192, 168)


In [14]:
#randomly occlude images in the same way as UNET

# Here we generate the occlusions and masks:

np.random.seed(42)   # we use this same random seed in all experiments to get identical masks

N, H, W = X.shape

X_occluded = X.copy()
masks = np.zeros_like(X)

for i in range(N):

    # generate random masks of size 15–30% of image size
    mask_h = np.random.randint(int(H * 0.15), int(H * 0.30))
    mask_w = np.random.randint(int(W * 0.15), int(W * 0.30))

    # generate random position to insert the mask.
    y = np.random.randint(0, H - mask_h)
    x = np.random.randint(0, W - mask_w)

    # apply mask to images. 
    X_occluded[i, y:y+mask_h, x:x+mask_w] = 0
    masks[i, y:y+mask_h, x:x+mask_w] = 1

print("Occluded dataset shape:", X_occluded.shape)
print("2414 images of dimension 192 by 168")

Occluded dataset shape: (2414, 192, 168)


In [30]:
# For UNet, we add channels. for RPCA, we flatten each 2D image into a 1D vector.
# The shape of the dataset will become (2414, 32256) (each row is an image)

X_occluded = X_occluded.reshape((X.shape[0], X.shape[1] * X.shape[2]))



#X_occluded = X_occluded.T
print(X_occluded.shape)

(2414, 32256)


In [ ]:
# Train and Test splits:
X_train_occ, X_test_occ /
X_train_clean, X_test_clean = train_test_split(
    X_occluded,
    X,
    test_size=0.2,
    random_state=42
)
print(X_train_occ.shape, X_test_occ.shape)

#RPCA expects shape with each column being an image. 
#Currently they are images as rows, so transpose

X_train_occ = X_train_occ.T
X_test_occ = X_test_occ.T
print(X_train_occ.shape, X_test_occ.shape) #1931 images for training, 483 for testing

X_train_clean = X_train_clean.T
X_test_clean = X_test_clean.T

In [31]:
#RPCA function

# http://kastnerkyle.github.io/posts/robust-matrix-decomposition/

def inexact_augmented_lagrange_multiplier(X, lmbda=.01, tol=1e-3,
                                          maxiter=100, verbose=True):
    """
    Inexact Augmented Lagrange Multiplier
    """
    Y = X
    norm_two = norm(Y.ravel(), 2)
    norm_inf = norm(Y.ravel(), np.inf) / lmbda
    dual_norm = np.max([norm_two, norm_inf])
    Y = Y / dual_norm
    A = np.zeros(Y.shape)
    E = np.zeros(Y.shape)
    dnorm = norm(X, 'fro')
    mu = 1.25 / norm_two
    rho = 1.5
    sv = 10.
    n = Y.shape[0]
    itr = 0
    while True:
        Eraw = X - A + (1 / mu) * Y
        Eupdate = np.maximum(Eraw - lmbda / mu, 0) + np.minimum(Eraw + lmbda / mu, 0)
        U, S, V = svd(X - Eupdate + (1 / mu) * Y, full_matrices=False)
        svp = (S > 1 / mu).shape[0]
        if svp < sv:
            sv = np.min([svp + 1, n])
        else:
            sv = np.min([svp + round(.05 * n), n])
        Aupdate = np.dot(np.dot(U[:, :svp], np.diag(S[:svp] - 1 / mu)), V[:svp, :])
        A = Aupdate
        E = Eupdate
        Z = X - A - E
        Y = Y + mu * Z
        mu = np.min([mu * rho, mu * 1e7])
        itr += 1
        if ((norm(Z, 'fro') / dnorm) < tol) or (itr >= maxiter):
            break
    if verbose:
        print("Finished at iteration %d" % (itr))  
    return A, E

In [ ]:
start_train = time.time()
L, S = inexact_augmented_lagrange_multiplier(X_train_occ)
#L is the low rank feature matrix, and S is the sparse noise matrix.
end_train = time.time()
print ("Time elapsed for robust PCA:", end_train - start_train, "s")

In [ ]:
start_train = time.time()

pca = PCA(n_components = 150)
pca.fit(L) #creates new basis of components to represent the clean faces
end_train = time.time()

X_test_occ_PCA = pca.transform(X_test_occ)


In [ ]:
#visualize occluded face with reconstructed faces

In [ ]:
#evaluation metrics
#compare reconstruction to clean images.(L_test vs X_test_clean)
print("MSE:", mse)
print("MAE:", mae)
print("PSNR:", psnr)
print("SSIM:", ssim_score)